# Joint Relative Astrometry and RV

```{note}
This tutorial requires some [extra packages](./about.ipynb) that are not included in the `jaxoplanet` dependencies.
```

The [radial velocity](https://jax.exoplanet.codes/en/latest/tutorials/rv/) and
[relative astrometry](./relative-astrometry.ipynb) tutorials
demonstrate how `jaxoplanet` can be used to fit RV and astrometric data, respectively.
In this tutorial, we will demonstrate how to model both simultaneously for the same system.
We will use observations from the $\beta$ Pictoris system to reproduce the results from [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30).

In [ ]:
import jax
import numpyro

# For multi-core parallelism (useful when running multiple MCMC chains in parallel)
numpyro.set_host_device_count(4)

# For CPU (use "gpu" for GPU)
numpyro.set_platform("cpu")

jax.config.update("jax_enable_x64", True)

## Relative astrometry data

We compiled the relative astrometry data in a single CSV with the `time` in MJD, the separation (`sep`) in mas, the position angle (`pa`) in degrees. The instrument is also given in an extra column.

In [ ]:
import pandas as pd
df_astro = pd.read_csv("https://zenodo.org/records/20312028/files/astrometry.csv")

df_astro.head()

To ensure the data works well with our JAX models below, let us first extract it to Numpy arrays.
We will also derive the right ascension (RA) and Declination (Dec) values for plotting purposes.

In [ ]:
import numpy as np

# Extract numpy arrays
t_astro = df_astro.time.values
sep = df_astro.sep.values
pa = df_astro.pa.values
sep_err = df_astro.sep_err.values
pa_err = df_astro.pa_err.values

# Convert to RA and Dec and derive uncertainties
pa_rad = np.radians(pa)
ra = sep * np.sin(pa_rad)
dec = sep * np.cos(pa_rad)
pa_err_rad = np.radians(pa_err)
dec_err = np.sqrt(
    (np.cos(pa_rad)**2 * sep_err**2) +
    (sep**2 * np.sin(pa_rad)**2 * pa_err_rad**2)
)
ra_err = np.sqrt(
    (np.sin(pa_rad)**2 * sep_err**2) +
    (sep**2 * np.cos(pa_rad)**2 * pa_err_rad**2)
)

# Store derived quantities in the dataframe
df_astro["ra"] = ra
df_astro["ra_err"] = ra_err
df_astro["dec"] = dec
df_astro["dec_err"] = dec_err

Let us now take a look at the RA and Dec, as this is usually more intuitive than Sep and PA.

In [ ]:
import matplotlib.pyplot as plt

def plot_radec():
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.errorbar(ra, dec, xerr=ra_err, yerr=dec_err, fmt="k.")
    ax.plot(0, 0, "k*", markersize=15)
    ax.axis("equal")
    ax.invert_xaxis()
    ax.set_xlabel("$\\Delta$ RA [mas]")
    ax.set_ylabel("$\\Delta$ Dec [mas]")
    return fig
_ = plot_radec()

And take a look at the corresponding Sep and PA coordinates as well since
this is what we will use to fit the data.

In [ ]:
def plot_seppa(fig=None):
    if fig is None:
        fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(6, 8))
    axs = fig.axes
    if len(axs) == 2:
        ax_sep, ax_pa = axs
    elif len(axs) == 4:
        ax_sep, ax_pa = axs[0], axs[2]
    ax_sep.errorbar(t_astro, sep, yerr=sep_err, fmt="k.")
    ax_sep.set_ylabel("Sep [mas]")
    ax_sep.set_ylim(0, None)
    ax_pa.errorbar(t_astro, pa, yerr=pa_err, fmt="k.")
    ax_pa.set_ylabel("PA [deg]")
    ax_pa.set_ylim(0, 360)

    axs[-1].set_xlabel("Time [MJD]")

    return fig
_ = plot_seppa()

## RV data

We also compiled the post-processed HARPS radial velocities from Table 3 of [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30) in a CSV.
The time is in MJD and the RVs are in m/s.

In [ ]:
df_rv = pd.read_csv("https://zenodo.org/records/20312028/files/rv.csv")
df_rv.head()

Again, we extract the values to Numpy arrays.

In [ ]:
t_rv = df_rv.mjd.values
rv = df_rv.rv.values
rv_err = df_rv.rv_err.values

And we take a look at the timeseries.

In [ ]:
def plot_rv(fig=None):
    if fig is None:
        fig, ax = plt.subplots()
    axs = fig.axes
    ax = axs[0]
    ax.errorbar(t_rv, rv, yerr=rv_err, fmt="k.")
    ax.set_ylabel("RV [m/s]")
    axs[-1].set_xlabel("Time [MJD]")
    return fig
_ = plot_rv()

## Forward model

We have two types of data, so we will need two separate forward models.
We could group everything under one model, but this would not be particularly efficient since the times are not the same for the two data types.

Instead, we will create a `build_system()` function that creates our `System` object for both forward models.

Since there are two planets in the $\beta$ Pic system, we add an optional flag to include planet c in the model.
We will only use this towards the end of the notebook.

In [ ]:
import jax.numpy as jnp

from jaxoplanet.orbits.keplerian import System, Body, Central
from jaxoplanet.types import Scalar

def build_system(params: dict[str, Scalar], fit_c: bool = False):
    # Build the system with beta Pic A and b
    system = System(
        # Radius is required by jaxoplanet but not used in the RV or astrometry models
        Central(mass=params["m_tot"], radius=1.0)
    ).add_body(
        Body(
            time_transit=params["Tc_b"],
            period=params.get("P_b", None),
            semimajor=params.get("a_b", None),
            inclination=params["i_b"],
            eccentricity=params["e_b"],
            omega_peri=params["omega_b"],
            sin_asc_node=jnp.sin(params["Omega_b"]),
            cos_asc_node=jnp.cos(params["Omega_b"]),
            parallax=params["parallax"],
            mass=params.get("m_b", None)
        )
    )
    if fit_c:
        # Add c if requested (RV parameters only)
        system = system.add_body(
            Body(
                time_transit=params["Tc_c"],
                period=params.get("P_c", None),
                semimajor=params.get("a_c", None),
                eccentricity=params["e_c"],
                omega_peri=params["omega_c"],
                mass=params.get("m_c", None)
            )
        )
    return system

A few things to note here:

- Masses are in units of `M_sun`
- Angles are in radians unless specified otherwise
- The semi-major axis is in units of `R_sun`
- The parallax is in arcsec

See the [units convention](../conventions.ipynb) for more on units and the [relative astrometry tutorial](./relative-astrometry.ipynb) for more on the astrometric convention.

### Astrometric model

The astrometric part of the model with compute separation and position angle at all provided times.
`jaxoplanet` returns them in arcsec and radians, so we convert to mas and degrees to be consistent with the data.

In [ ]:
from jax.typing import ArrayLike

deg = np.pi / 180

def astrometry_model(time: ArrayLike, params: dict[str, Scalar]):
    system = build_system(params)
    sep, pa = system.bodies[0].relative_angles(time, parallax=params["parallax"])
    sep = sep * 1e3
    pa = pa / deg
    return sep, pa % 360

We can now test the model with manually selected parameter values.
We picked values not too far from the priors and posteriors of the paper to test the forward model and make sure it works as expected.

In [ ]:
import astropy.units as u
from jaxoplanet import constants

yr = 365.25

init_params = {
    "a_b": 8.5 * constants.au,
    "Tc_b": 58010,
    "i_b": 89 * deg,
    "e_b": 0.0,
    "omega_b": 13 * deg,
    "Omega_b": 32 * deg,
    "parallax": 51.44 * 1e-3,
    "m_tot": 1.6,
    "m_b": 10.0 * constants.M_jup,
}

In [ ]:
fig, axs = plt.subplots(nrows=4, sharex=True, figsize=(6, 8), gridspec_kw={"height_ratios": (2, 1, 2, 1)})

t_astro_fine = np.linspace(t_astro.min() - 10 * yr, t_astro.max() + 10 * yr, num=1000)
sep_mod, pa_mod = astrometry_model(t_astro, init_params)
sep_fine, pa_fine = astrometry_model(t_astro_fine, init_params)

def plot_seppa_mod(fig, mod_dict):
    axs = fig.axes
    sep_mod = mod_dict["sep_mod"]
    pa_mod = mod_dict["pa_mod"]
    sep_fine = mod_dict["sep_fine"]
    pa_fine = mod_dict["pa_fine"]
    axs[0].plot(t_astro_fine, sep_fine)
    axs[1].errorbar(t_astro, sep - sep_mod, yerr=sep_err, fmt="k.")
    axs[1].set_ylabel("Sep residuals [mas]")
    axs[1].axhline(0, linestyle="--")
    axs[2].plot(t_astro_fine, pa_fine)
    axs[3].errorbar(t_astro, pa - pa_mod, yerr=pa_err, fmt="k.")
    axs[3].set_ylabel("PA residuals [deg]")
    axs[3].axhline(0, linestyle="--")
    _, ymax = axs[0].get_ylim()
    sep_max = max(sep_fine.max() * 1.05, ymax)
    axs[0].set_ylim(0, sep_max)
    return fig

init_mods = {"sep_fine": sep_fine, "pa_fine": pa_fine, "sep_mod": sep_mod, "pa_mod": pa_mod}
plot_seppa(fig)
plot_seppa_mod(fig, init_mods)

_ = axs[0].set_title("Initial model for $\\beta$ Pic b relative astrometry")

In [ ]:
def plot_radec_model(fig, mod_dict):
    ax = fig.axes[0]
    sep_fine = mod_dict["sep_fine"]
    pa_fine = mod_dict["pa_fine"]
    ra_fine = sep_fine * np.sin(pa_fine * deg)
    dec_fine = sep_fine * np.cos(pa_fine * deg)
    ax.plot(ra_fine, dec_fine)
fig = plot_radec()
plot_radec_model(fig, init_mods)
fig.axes[0].set_title("Initial model for $\\beta$ Pic b")
plt.tight_layout()

Not too bad for a first guess!

### RV model

`jaxoplanet`'s `System.radial_velocity()` function returns the star's RVs in `Rsun/day`,
which we convert to `m/s` as most RV data is specified in these units.
Luckily the `jaxoplanet.constants` module can help with that as we show in the function below.

In [ ]:
def rv_model(time: ArrayLike, params: dict[str, Scalar], total: bool = True, fit_c: bool = False):
    """Returns the stellar RVs in m/s

    Args:
        time: Time values
        params: dictionary of system parameters
        total: Return the total RV if True and the per-planet signals otherwise,
               defaults to True
        fit_c: Include beta Pic c in the model if True, defaults to False
    Returns:
        Array of RV values with shape ``(time.size,)`` or ``(nplanets, time.size)``
    """
    system = build_system(params, fit_c=fit_c)
    rvs = system.radial_velocity(time) / constants.m_per_s
    if total:
        rvs = rvs.sum(axis=0)
    return rvs

Let us generate a model accounting only for $\beta$ Pic b.
We will include c later in the notebook

In [ ]:
t_rv_fine = np.linspace(t_rv.min(), t_rv.max(), num=1000)
rv_fine = rv_model(t_rv_fine, init_params)
rv_mod = rv_model(t_rv, init_params)
init_mods["rv_mod"] = rv_mod
init_mods["rv_fine"] = rv_fine

def plot_rv_mod(fig, mod_dict):
    axs = fig.axes
    rv_mod = mod_dict["rv_mod"]
    rv_fine = mod_dict["rv_fine"]
    axs[0].plot(t_rv_fine, rv_fine)
    axs[1].errorbar(t_rv, rv - rv_mod, yerr=rv_err, fmt="k.")
    axs[1].axhline(0, linestyle="--")
    axs[1].set_ylabel("RV residuals [m/s]")

fig, axs = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": (2, 1)})
plot_rv(fig=fig)
plot_rv_mod(fig, init_mods)
_ = axs[0].set_title("Initial RV Model for $\\beta$ Pic b")

## Probabilistic model

To sample this forward model, we will need to wrap it in a Numpyro model.
We implement the same priors as those from Table 2 of [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30).

We will start by running the model with only $\beta$ Pic b and without the GRAVITY data,
but options for $\beta$ Pic b+c and GRAVITY are already implemented in our model here
since we re-use the same function later in the tutorial.

The comments provide more explanation of each section in the model.


In [ ]:
import numpyro
from numpyro import distributions as dist, infer
from numpyro_ext import distributions as distx

def model(fit_c: bool = False, inflate: bool = False, gravity: bool = False):
    # Time of conjunction uniform in MJD
    tc_b = numpyro.sample("tc_b", dist.Uniform(57000, 59000))

    # h=sqrt(e)*cos(omega), k=sqrt(e)*sin(omega)
    hk_b = numpyro.sample("hk_b", distx.UnitDisk())
    h_b, k_b = hk_b[..., 0], hk_b[..., 1]
    numpyro.deterministic("h_b", h_b)
    numpyro.deterministic("k_b", k_b)
    e_b = numpyro.deterministic("e_b", h_b**2 + k_b**2)
    omega_b = numpyro.deterministic("omega_b", jnp.arctan2(k_b, h_b))

    # Prior uniform in log-a between 1 and 100 AU. Convert to R_sun below
    loga_b = numpyro.sample("loga_b", dist.Uniform(jnp.log(1), jnp.log(100)))
    a_b = numpyro.deterministic("a_b", jnp.exp(loga_b))

    # Mass of beta pic b in M_jup, convert to M_sun below
    m_b = numpyro.sample("m_b", dist.Uniform(1, 20))

    Omega_b = numpyro.sample("Omega_b", dist.Uniform(18 * deg, 90 * deg))

    # equivalent to the sin(i) prior in the paper
    cosi_b = numpyro.sample("cosi_b", dist.Uniform(-1, 1))
    i_b = numpyro.deterministic("i_b", jnp.arccos(cosi_b))

    # Parallax in arcsec
    plx = numpyro.sample("plx", dist.Normal(51.44e-3, 0.12e-3))

    # RV offset in m/s
    gamma = numpyro.sample("gamma", dist.Uniform(-100, 100))

    # Total mass of the system in M_sun
    m_tot = numpyro.sample("m_tot", dist.Uniform(1.4, 2.0))

    # Note the conversion M_jup->M_sun for mass and AU->R_sun for a
    params = {
        "m_tot": m_tot,
        "m_b": m_b * constants.M_jup,
        "Tc_b": tc_b,
        "a_b": a_b * constants.au,
        "e_b": e_b,
        "i_b": i_b,
        "omega_b": omega_b,
        "Omega_b": Omega_b,
        "parallax": plx,
    }

    if fit_c:
        # Add the parameters for beta Pic c if requested

        # Time of conjunction uniform in MJD
        tc_c = numpyro.sample("tc_c", dist.Uniform(57500, 59000))

        # h=sqrt(e)*cos(omega), k=sqrt(e)*sin(omega)
        hk_c = numpyro.sample("hk_c", distx.UnitDisk())
        h_c, k_c = hk_c[..., 0], hk_c[..., 1]
        numpyro.deterministic("h_c", h_c)
        numpyro.deterministic("k_c", k_c)
        e_c = numpyro.deterministic("e_c", h_c**2 + k_c**2)
        omega_c = numpyro.deterministic("omega_c", jnp.arctan2(k_c, h_c))

        # Prior uniform in log-a between 1 and 100 AU. Convert to R_sun below
        loga_c = numpyro.sample("loga_c", dist.Uniform(jnp.log(2), jnp.log(3.5)))
        a_c = numpyro.deterministic("a_c", jnp.exp(loga_c))

        # Mass of beta pic b in M_jup, convert to M_sun below
        msini_c = numpyro.sample("msini_c", dist.Uniform(1, 20))

        params = params | {
            "m_c": msini_c * constants.M_jup,
            "Tc_c": tc_c,
            "a_c": a_c * constants.au,
            "e_c": e_c,
            "omega_c": omega_c,
        }

    # Save model output at observation time for the likelihood
    rv_mod = rv_model(t_rv, params, total=False, fit_c=fit_c) - gamma
    sep_mod, pa_mod = astrometry_model(t_astro, params)

    # Inflate the error bars if requested
    if inflate:
        log_sep_s = numpyro.sample("log_sep_s", dist.Normal(loc=np.log(np.median(sep_err)), scale=2.0))
        log_pa_s = numpyro.sample("log_pa_s", dist.Normal(loc=np.log(np.median(pa_err)), scale=2.0))
        log_rv_s = numpyro.sample("log_rv_s", dist.Normal(loc=np.log(np.median(rv_err)), scale=2.0))
        sep_tot_err = jnp.sqrt(sep_err**2 + jnp.exp(2 * log_sep_s))
        pa_tot_err = jnp.sqrt(pa_err**2 + jnp.exp(2 * log_pa_s))
        rv_tot_err = jnp.sqrt(rv_err**2 + jnp.exp(2 * log_rv_s))
    else:
        sep_tot_err = sep_err
        pa_tot_err = pa_err
        rv_tot_err = rv_err

    # Likelihood on sep and pa
    # The likelihood for pa accounts for wrapping of the angle
    numpyro.sample("sep_obs", dist.Normal(loc=sep_mod, scale=sep_tot_err), obs=sep)
    numpyro.sample("rv_obs", dist.Normal(loc=rv_mod.sum(axis=0), scale=rv_tot_err), obs=rv)
    pa_diff = jnp.arctan2(
        jnp.sin(pa_mod * deg - pa * deg), jnp.cos(pa_mod * deg - pa * deg)
    )
    numpyro.sample("pa_obs", dist.Normal(loc=pa_diff, scale=pa_tot_err * deg), obs=0.0)

    numpyro.deterministic("sep_mod", sep_mod)
    numpyro.deterministic("pa_mod", pa_mod)
    numpyro.deterministic("rv_mod", rv_mod.sum(axis=0))

    # Add a likelihood for the GRAVITY data if requested
    if gravity:
        sep_mod_g, pa_mod_g = astrometry_model(t_gravity, params)
        pa_rad = pa_mod_g[0] * np.pi / 180
        ra_mod_g = sep_mod_g[0] * jnp.sin(pa_rad)
        dec_mod_g = sep_mod_g[0] * jnp.cos(pa_rad)
        gravity_mean = jnp.array([ra_mod_g, dec_mod_g])
        gravity_C = jnp.array([[ra_var_gravity, cov_gravity], [ cov_gravity, dec_var_gravity]])
        numpyro.sample("gravity_obs", dist.MultivariateNormal(loc=gravity_mean, covariance_matrix=gravity_C), obs=jnp.array([ra_gravity, dec_gravity]))

    # Save finer grid for plotting
    sep_dense, pa_dense = astrometry_model(t_astro_fine, params)
    rv_dense_arr = rv_model(t_rv_fine, params, total=False, fit_c=fit_c) - gamma
    rv_dense = rv_dense_arr.sum(axis=0)
    rv_dense_b = rv_dense_arr[0]
    numpyro.deterministic("sep_fine", sep_dense)
    numpyro.deterministic("pa_fine", pa_dense)
    numpyro.deterministic("rv_fine", rv_dense)
    numpyro.deterministic("rv_fine_b", rv_dense_b)

### Prior predictive

It is always a good idea to check the prior in parameter space and in model space before going further with the inference.

Let us start by generating prior samples with NumPyro.

In [ ]:
n_prior_samples = 2000

key = jax.random.key(0)

key, subkey = jax.random.split(key)
prior_samples = infer.Predictive(model, num_samples=n_prior_samples)(subkey)

To futher manipulate the samples, we will use the Arviz library.
Arviz can import samples in a dictionary, but we need to add a "chain" dimension as the first axis of each array.

We also specify which variables we want to show in the plots to avoid showing all model outputs, for example.

In [ ]:
import arviz as az
converted_prior_samples = {
    f"{p}": np.expand_dims(prior_samples[p], axis=0) for p in prior_samples
}
fit_names = ["tc_b", "h_b", "k_b", "loga_b", "m_b", "Omega_b", "cosi_b", "plx", "gamma", "m_tot"]
det_names = ["e_b", "omega_b", "a_b", "i_b"]
var_names= fit_names + det_names
prior_idata = az.from_dict(prior=converted_prior_samples)

Now that our prior is stored in arviz, we can display the parameter distributions with `corner`.

In [ ]:
import corner

_ = corner.corner(prior_idata, group="prior", var_names=var_names)

We can also visualize model outputs corresponding to parameter sets in the prior.

In [ ]:
# Pick a subset of indices to plot
rng = np.random.default_rng()

def plot_seppa_samples(fig, samples, n_plots=100):
    axs = fig.axes
    plot_inds = rng.integers(n_prior_samples, size=n_plots)
    sep_max = 0.0
    for idx in plot_inds:
        sep_fine = samples["sep_fine"].values.reshape(-1, t_astro_fine.size)[idx]
        sep_max = max(sep_max, sep_fine.max())
        axs[0].plot(t_astro_fine, sep_fine, "C1", alpha=0.1)
        axs[1].plot(t_astro_fine, samples["pa_fine"].values.reshape(-1, t_astro_fine.size)[idx], "C1", alpha=0.1)
    axs[0].relim()
    axs[0].autoscale_view()
    axs[0].set_ylim(0, sep_max)

fig = plot_seppa()
_ = plot_seppa_samples(fig, prior_idata.prior)

In [ ]:
def plot_radec_samples(fig, samples, n_plots=100):
    ax = fig.axes[0]
    plot_inds = rng.integers(n_prior_samples, size=n_plots)
    for idx in plot_inds:
        sep_fine = samples["sep_fine"].values.reshape(-1, t_astro_fine.size)[idx]
        pa_fine = samples["pa_fine"].values.reshape(-1, t_astro_fine.size)[idx]
        ra_fine = sep_fine * np.sin(pa_fine * deg)
        dec_fine = sep_fine * np.cos(pa_fine * deg)
        ax.plot(ra_fine, dec_fine, "C1", alpha=0.1)
fig = plot_radec()
_ = plot_radec_samples(fig, prior_idata.prior)

In [ ]:
def plot_rv_samples(fig, samples, n_plots=100):
    ax = fig.axes[0]
    plot_inds = rng.integers(n_prior_samples, size=n_plots)
    for idx in plot_inds:
        ax.plot(t_rv_fine, samples["rv_fine"].values.reshape(-1, t_rv_fine.size)[idx], "C1", alpha=0.1)
fig = plot_rv()
_ = plot_rv_samples(fig, prior_idata.prior)

### Optimization

Now that we know our prior encompasses the data as expected, we can optimize the model.
The `numpyro_ext` module provides various utility functions, including one to optimize a
NumPyro model.

In [ ]:
import numpyro_ext.optim as optimx

key, subkey = jax.random.split(key)
map_soln = optimx.optimize(model, init_strategy=infer.init_to_median())(subkey)

In [ ]:
fig, axs = plt.subplots(nrows=4, sharex=True, figsize=(6, 8), gridspec_kw={"height_ratios": (2, 1, 2, 1)})
plot_seppa(fig)
plot_seppa_mod(fig, map_soln)
_ = axs[0].set_title("Optimal model for $\\beta$ Pic b relative astrometry")

In [ ]:
fig = plot_radec()
plot_radec_model(fig, map_soln)
fig.axes[0].set_title("Optimal model for $\\beta$ Pic b")
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": (2, 1)})
plot_rv(fig=fig)
plot_rv_mod(fig, map_soln)
_ = axs[0].set_title("Optimal RV Model for $\\beta$ Pic b")

In [ ]:
import pprint
print("MAP Solution")
pprint.pprint({v: map_soln[v].item() for v in var_names})

### Sampling

We will now sample the model to understand the posterior distribution
and derive uncertainties on our parameters.
We use NumPyro's No-U-Turn Sampler (NUTS) implementation for this.

In [ ]:
sampler = infer.MCMC(
    infer.NUTS(
        model,
        dense_mass=True,
        regularize_mass_matrix=True,
        init_strategy=infer.init_to_value(values=map_soln)
    ),
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=True,
)
key, subkey = jax.random.split(key)
sampler.run(subkey)

Arviz can the automatically load the posterior from NumPyro and summarize it.

In [ ]:
idata = az.from_numpyro(sampler)
az.summary(idata, var_names=var_names)

We can also visualize the posterior with `corner`.

In [ ]:
_ = corner.corner(idata, var_names=var_names)

And we can again draw model samples.

In [ ]:
fig = plot_seppa()
_ = plot_seppa_samples(fig, idata.posterior)

In [ ]:
fig = plot_radec()
_ = plot_radec_samples(fig, idata.posterior)

In [ ]:
fig = plot_rv()
_ = plot_rv_samples(fig, idata.posterior)

## Adding Beta Pic c

There is actually a second planet in this system, $\beta$ Pic c.
In the dataset we have here, it is only detectable through radial velocities.

The `model()` function enables us to account for c through the `fit_c` flag.
We need to pass this to the optimization and MCMC

Let us first add parameters for planet c to our initial guess.
Here we give the parameter names that `rv_model()` and `build_model()` expect.

In [ ]:
init_params_c = {
    "a_c": 3.0 * constants.au,
    "Tc_c": 57800,
    "e_c": 0.0,
    "omega_c": -50 * deg,
    "m_c": 5.0 * constants.M_jup,
}
init_params_both = init_params | init_params_c

Only the RV model should be affected. Let us take a look.

In [ ]:
rv_fine = rv_model(t_rv_fine, init_params_both, fit_c=True)
rv_mod = rv_model(t_rv, init_params_both, fit_c=True)
init_mods_both = {}
init_mods_both["rv_mod"] = rv_mod
init_mods_both["rv_fine"] = rv_fine

fig, axs = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": (2, 1)})
_ = plot_rv(fig=fig)
_ = plot_rv_mod(fig, init_mods_both)

The 2-planet model is a bit trickier to optimize, so we will give start parameters that match
the MAP solution for b and take an educated guess for c.

In [ ]:
# Setup map_b to be a proper input
map_b = {v: map_soln[v].item() for v in fit_names}
map_b["hk_b"] = map_soln["hk_b"]
del map_b["h_b"]
del map_b["k_b"]
start_c = {
    "hk_c": jnp.array([0.01, 0.01]),
    "tc_c": 57800.0,
    "msini_c": 8.0,
    "loga_c": np.log(2.5),
}
init_params_opt = map_b | start_c

print(f"Optimizing the model")
init = infer.init_to_value(values=init_params_opt)
key, subkey = jax.random.split(key)
map_soln_both = optimx.optimize(model, init_strategy=init)(subkey, fit_c=True)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={"height_ratios": (2, 1)})
plot_rv(fig=fig)
plot_rv_mod(fig, map_soln_both)
_ = axs[0].set_title("Optimal RV Model for $\\beta$ Pic b+c")

And again we can sample our model with NumPyro.

In [ ]:
sampler = infer.MCMC(
    infer.NUTS(
        model,
        dense_mass=True,
        regularize_mass_matrix=True,
        init_strategy=infer.init_to_value(values=map_soln_both)
    ),
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=True,
)
key, subkey = jax.random.split(key)
sampler.run(subkey, fit_c=True)

In [ ]:
idata = az.from_numpyro(sampler)

In [ ]:
fit_names_both = fit_names + ["tc_c", "h_c", "k_c", "loga_c", "msini_c"]
det_names_both = det_names + ["e_c", "omega_c", "a_c"]
var_names_both = fit_names_both + det_names_both
az.summary(idata, var_names=var_names_both)

In [ ]:
_ = corner.corner(idata, var_names=var_names_both)

And we can again draw model samples.

In [ ]:
fig = plot_seppa()
_ = plot_seppa_samples(fig, idata.posterior)

In [ ]:
fig = plot_radec()
_ = plot_radec_samples(fig, idata.posterior)

In [ ]:
fig = plot_rv()
_ = plot_rv_samples(fig, idata.posterior)

## Including GRAVITY data

The last thing we are missing to replicate the results from the paper is the inclusion of the GRAVITY data point.
This requires a multivariate (correlated) likelihood in RA and DEC and is implemented in the model when `gravity=True`.
We do not need to add any parameter to the model: we only load the data and pass the `gravity` flag

In [ ]:
df_gravity = pd.read_csv("https://zenodo.org/records/20312028/files/gravity.csv")
df_gravity

In [ ]:
t_gravity = df_gravity.time.values
ra_gravity = df_gravity.ra.values.item()
ra_var_gravity = df_gravity.ra_var.values.item()
dec_gravity = df_gravity.dec.values.item()
dec_var_gravity = df_gravity.dec_var.values.item()
cov_gravity = df_gravity.radec_cov.values.item()

Since there are no new parameters, we can try to re-use our MAP solution from above and go straight the sampling phase.
The GRAVITY data point is very precise and thus quite constraining.
This seems to make the HMC sampler a bit slower to converge.
We keep it short here since this is a tutorial, but feel free to increase it.
Setting `num_warmup=2000`, `num_samples=2000` and `num_chains=4` seems to already give better results and
should take around 10 minutes on a laptop.

In [ ]:
mcmc_start = map_soln_both.copy()
mcmc_start["m_b"] = 10.0  # mass is sometimes mis-behaving in optimizer
sampler = infer.MCMC(
    infer.NUTS(
        model,
        dense_mass=True,
        regularize_mass_matrix=True,
        init_strategy=infer.init_to_value(values=mcmc_start)
    ),
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=True,
)
key, subkey = jax.random.split(key)
sampler.run(subkey, fit_c=True, gravity=True)

In [ ]:
idata_no_gravity = idata.copy()
idata = az.from_numpyro(sampler)
az.summary(idata, var_names=var_names_both)

And we can again draw model samples.

In [ ]:
fig = plot_seppa()
_ = plot_seppa_samples(fig, idata.posterior)

In [ ]:
fig = plot_radec()
_ = plot_radec_samples(fig, idata.posterior)

In [ ]:
fig = plot_rv()
_ = plot_rv_samples(fig, idata.posterior)

We can also compare the posterior distributions and see how the GRAVITY data affects them.

In [ ]:
fig = corner.corner(idata, var_names=var_names_both, color="b")
_ = corner.corner(idata_no_gravity, var_names=var_names_both, fig=fig, color="r")

## Inflating the errors

One thing that was not done by [Vandal et al. (2020)](https://iopscience.iop.org/article/10.3847/1538-3881/abba30)
is to add an inflation term to the astrometric and RV errors.
Let us see how this affects the results.

In [ ]:
var_names_both_no_s = var_names_both
var_names_both = var_names_both + ["log_sep_s", "log_pa_s", "log_rv_s"]

In [ ]:
print(f"Optimizing the model")
init = infer.init_to_value(values=init_params_opt)
key, subkey = jax.random.split(key)
map_soln_both = optimx.optimize(model, init_strategy=init)(subkey, fit_c=True, inflate=True, gravity=True)

In [ ]:
mcmc_start = map_soln_both.copy()
mcmc_start["m_b"] = 10.0  # mass is sometimes mis-behaving in optimizer
sampler = infer.MCMC(
    infer.NUTS(
        model,
        dense_mass=True,
        regularize_mass_matrix=True,
        init_strategy=infer.init_to_value(values=mcmc_start)
    ),
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=True,
)
key, subkey = jax.random.split(key)
sampler.run(subkey, fit_c=True, gravity=True, inflate=True)

In [ ]:
idata_no_s = idata.copy()
idata = az.from_numpyro(sampler)
az.summary(idata, var_names=var_names_both)

As we can see from the summary above, the error terms on separation and RV are non-negligible,
and adding this flexibility to the models inflates the uncertainties in other parameters quite significantly (the mass, for example).
We can also visualize the differences in a corner plot.

In [ ]:
fig = corner.corner(idata, var_names=var_names_both_no_s, color="b")
_ = corner.corner(idata_no_s, var_names=var_names_both_no_s, fig=fig, color="r")

And we can again draw model samples.

In [ ]:
fig = plot_seppa()
_ = plot_seppa_samples(fig, idata.posterior)

In [ ]:
fig = plot_radec()
_ = plot_radec_samples(fig, idata.posterior)

In [ ]:
fig = plot_rv()
_ = plot_rv_samples(fig, idata.posterior)